In [29]:
from langchain_classic.document_loaders import TextLoader,DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser

In [30]:
loader = DirectoryLoader(r'C:\Users\mendh\Langchain-Langgraph\0-Dataingestion-parsing\data\text_files', glob='**/*.txt', show_progress=True,loader_cls=TextLoader)
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(documents)
print(f"Number of documents: {len(docs)}")

100%|██████████| 4/4 [00:00<00:00, 1811.99it/s]

Number of documents: 14


In [31]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
retriver = vectorstore.as_retriever(search_kwargs={"k": 10})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["Groq_API_Key"] = os.getenv("Groq_API_Key")
llm = init_chat_model(model_provider="groq", model="llama-3.1-8b-instant")
llm.invoke("Hello, how are you?")

AIMessage(content="I'm functioning properly, thank you for asking. I'm a large language model, so I don't have emotions or feelings like humans do, but I'm here and ready to help with any questions or tasks you have. How can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 41, 'total_tokens': 94, 'completion_time': 0.064588728, 'completion_tokens_details': None, 'prompt_time': 0.002061011, 'prompt_tokens_details': None, 'queue_time': 0.047096694, 'total_time': 0.066649739}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4352-7b5c-7e50-b2d5-061ed9ac4acc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 53, 'total_tokens': 94})

In [33]:
prompt = PromptTemplate.from_template(
    """you are a helpful assistant.your task is to rank the following documentsfrom most to least relevant to the question.
Question: {input}
Documents:
{documents}

Instructions:
Think about the relevance of each document to the user's question.
return a list of document indicies in ranked order, with the most relevant document first.

output format:
(0,1,2,3,4)
""")

In [34]:
chain = prompt|llm|StrOutputParser()
chain

PromptTemplate(input_variables=['documents', 'input'], input_types={}, partial_variables={}, template="you are a helpful assistant.your task is to rank the following documentsfrom most to least relevant to the question.\nQuestion: {input}\nDocuments:\n{documents}\n\nInstructions:\nThink about the relevance of each document to the user's question.\nreturn a list of document indicies in ranked order, with the most relevant document first.\n\noutput format:\n(0,1,2,3,4)\n")
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002C533598A50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002C533599450>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('*****

In [35]:
retrived_docs = retriver.invoke("What is Machine Learning?")
docs = chain.invoke({"input": "What is Machine Learning?", "documents": retrived_docs})

In [36]:
print(docs)

To determine the most relevant documents, I'll analyze the content of each document and rank them based on their relevance to the question "What is Machine Learning?"

Here's the ranking:

1. Document(id='2d71ce82-4a48-4124-a069-166943763139'): This document directly defines machine learning as a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data.

2. Document(id='e0f33ec3-9be1-4c55-a738-b65cc8b37a1c'): This document provides a formal definition of machine learning and explains how it is concerned with improving performance at tasks in a class of tasks.

3. Document(id='711ede74-8f7c-4503-827f-3f12a8b56de0'): This document explains the relationships between machine learning, artificial intelligence, and deep learning, which is relevant to understanding the broader context of machine learning.

4. Document(id='04773836-53d0-4154-8f4e-fd31a9189b14'): This document provides a h